# AEQGA: calibrated cosmological parameter estimation

This notebook uses the current project modules to run the workflow described in [Sarracino et al., arXiv:2602.15459v1](https://arxiv.org/html/2602.15459v1). Parameters are always ordered **[H0, Omega_m]**.

The default is a short **demonstration**, not a reproduction of the paper's 300-run statistics. SNe estimation uses **Pantheon+ calibrated MU_SH0ES and the full statistical+systematic covariance**, with zHD in the distance integral and zHEL in the luminosity prefactor. No mock problem, legacy Pantheon loader, arbitrary fixed absolute magnitude, or magnitude marginalization is used.

Workflow: configuration → real-data checks → classical reference → selection → amplitude encoding → crossover/mutation → final measurement/decoding → AEQGA evolution → diagnostics → optional independent-run ensemble. **CMB+BAO is a separate experiment, not an all-probe sum.**

The old mock/uncalibrated results, failed KDE output and stale execution logs have been removed. See [the replication specification](../docs/REPLICATION_SPEC.md) for unresolved paper conventions.

## 1. Setup and imports

From the repository root, install the package and obtain the SNe data before running all cells:

```bash
python -m pip install -e .
git clone https://github.com/CobayaSampler/sn_data.git sn_data
git -C sn_data checkout 61d96434cafc2770928322c38e5a750e686368ae
# Only needed for the optional CMB experiment:
python -m scripts.fetch_cmb_data
```

This notebook does not download data automatically and never substitutes a mock likelihood when data are missing.

In [ ]:
import hashlib
import json
import random
import sys
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path

# Works when Jupyter starts in the repository root or notebooks/.
candidates = [Path.cwd(), *Path.cwd().parents]
root = next((p for p in candidates if (p / "pyproject.toml").is_file()), None)
if root is not None and str(root) not in sys.path:
    sys.path.insert(0, str(root))

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from IPython import get_ipython
get_ipython().run_line_magic('matplotlib', 'inline')
from qiskit import transpile
from qiskit.quantum_info import Statevector
from qiskit_aer import AerSimulator

from aeqga.paths import PROJECT_ROOT, output_path
from aeqga.likelihoods.pantheon_problem import PantheonPlusProblem
from aeqga.likelihoods.bao_cmb_problem import BAOLikelihood, BAOCMBProblem
from aeqga.steps.selection.population import split_population
from aeqga.steps.encoding.amplitude_encoding import build_amplitude_circuit
from aeqga.steps.decoding.measurement_decoding import (
    get_basis_counts, decode_random_subset, decode_elite_subset,
)
from aeqga.steps.evolution.aeqga_algorithm import (
    AEQGAParameters, build_dimension_circuit, run_aeqga_dual, run_aeqga_sv,
)

plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 220, "font.family": "DejaVu Sans",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": False, "axes.labelsize": 11, "axes.titlesize": 12,
})
VERSIONS = {name: version(name) for name in ["numpy", "scipy", "qiskit", "qiskit-aer", "matplotlib"]}
print("Project:", PROJECT_ROOT)
print("Versions:", VERSIONS)

## 2. Configuration: demonstration versus paper settings

Choose `MODE = "paper"` for 32 individuals and 50 updates per run. The ensemble uses 300 runs only when explicitly enabled. The demonstration uses 16 individuals and 10 updates; it cannot establish published uncertainties.

4096 shots is a recorded local choice, not a value established by the paper. Distance-grid sizes 100/300 reproduce the paper's nearest-neighbor approximation; 0 selects direct integration. A statevector run is an optional diagnostic that removes measurement shot noise, not all algorithmic randomness.

In [ ]:
MODE = "demo"                 # "demo" or "paper"
BASE_SEED = 23
DISTANCE_GRID = 300           # 0 = direct integration; 100/300 = nearest neighbor
SELECTION = "all"             # "all" or "hubble_flow"
REDSHIFT = "hd_hel"           # zHD integral, zHEL luminosity prefactor
RUN_STATEVECTOR = False
RUN_ENSEMBLE = False
RUN_CMB_BAO = False
RUN_HQGA_COMPARISON = False
HQGA_PARAMS = None            # Supply native HQGA parameters in a compatible environment
HQGA_DEVICE_FEATURES = None   # Supply native HQGA device configuration
CMB_BACKEND = "camb"           # explicit reference; "pico" is strict, "hybrid" records fallbacks
CMB_ELL_MIN = 2
CMB_ERROR_MODE = "average"

if MODE not in {"demo", "paper"}:
    raise ValueError("MODE must be demo or paper")
settings = {"demo": (16, 10, 1024, 20), "paper": (32, 50, 4096, 300)}
POP_SIZE, MAX_GEN, SHOTS, ENSEMBLE_RUNS = settings[MODE]
params = AEQGAParameters(
    pop_size=POP_SIZE, max_gen=MAX_GEN, num_shots=SHOTS,
    p_cross=0.5, p_mut=0.5, elite_margin=0.0,
    progress_bar=False,
)
params._validate()

def seed_run(seed, *, statevector=False):
    random.seed(seed)
    np.random.seed(seed)
    return AerSimulator(method="statevector" if statevector else "automatic",
                        seed_simulator=seed)

print(f"{MODE}: population={POP_SIZE}, updates={MAX_GEN}, shots={SHOTS}")
print(f"Expected objective evaluations per run: {POP_SIZE * (MAX_GEN + 1)}")
print("Expensive opt-ins:", dict(statevector=RUN_STATEVECTOR, ensemble=RUN_ENSEMBLE,
                                 cmb_bao=RUN_CMB_BAO))

## 3. Load and audit Pantheon+

Retain light-curve rows rather than deduplicating source IDs. The loader applies any row mask to both covariance axes and factors the total covariance once; do not add diagonal statistical variances again. Published decimal asymmetry is symmetrized and recorded.

Full covariance chi-squared uses calibrated observed distance moduli. Marginalizing an arbitrary magnitude offset would erase the H0 dependence and invalidate this two-parameter fit.

In [ ]:
SN_DIRECTORY = PROJECT_ROOT / "sn_data" / "PantheonPlus"
required_sn = [
    SN_DIRECTORY / "Pantheon+SH0ES.dat",
    SN_DIRECTORY / "Pantheon+SH0ES_STAT+SYS.cov",
]
missing = [str(p) for p in required_sn if not p.is_file()]
if missing:
    raise FileNotFoundError("Missing Pantheon+ inputs. Follow section 1; no mock fallback. " + str(missing))
sne_problem = PantheonPlusProblem(
    SN_DIRECTORY, selection=SELECTION, redshift=REDSHIFT, grid_size=DISTANCE_GRID,
)
data = sne_problem._data

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

DATA_PROVENANCE = {p.name: sha256_file(p) for p in required_sn}
assert data["cov_total"].shape == (data["n_sn"], data["n_sn"])
assert np.all(np.diag(data["cholesky"]) > 0)
if SELECTION == "all":
    assert data["n_sn"] == 1701
print(json.dumps({
    "light_curve_rows": data["n_sn"],
    "selection": data["selection"], "redshift": data["redshift"],
    "calibrator_rows": int(np.sum(data["is_calibrator"])),
    "covariance_max_asymmetry": data["covariance_max_asymmetry"],
    "distance_grid": DISTANCE_GRID,
    "data_sha256": DATA_PROVENANCE,
}, indent=2))

## 4. Independent classical reference

Profile the H0 distance-modulus shift and search Omega_m independently of AEQGA. Compare the configured approximation with direct distances. Paper values are comparison markers only, not part of the objective or initialization target. Differences in row selection/redshift conventions must be reported rather than tuned away.

In [ ]:
configured_reference = sne_problem.classical_minimum()
if DISTANCE_GRID == 0:
    direct_problem = sne_problem
else:
    direct_problem = PantheonPlusProblem(
        SN_DIRECTORY, selection=SELECTION, redshift=REDSHIFT, grid_size=0,
    )
direct_reference = direct_problem.classical_minimum()
paper_sne_point = np.array([72.82, 0.363])
print("Configured reference:", configured_reference)
print("Direct reference:", direct_reference)
print("Configured chi2 at paper point:", sne_problem.compute_fitness(paper_sne_point))
print("H0 sensitivity at fixed Omega_m:",
      {h: sne_problem.compute_fitness([h, configured_reference["Omega_m"]])
       for h in [65., configured_reference["H0"], 80.]})

## 5. Inspect one generation's quantum workflow

The elite quarter bypasses all circuits. Copies of elites and fresh random individuals enter separate circuits for **each** parameter. Crossover consists of two oppositely controlled CRY(pi/2) gates; mutation is RX(pi/2). Measurement comes last.

The illustration below forces both gate types on to verify ordering; the optimization itself uses probabilities 0.5. Random decoding is count min–max scaling; elite-copy decoding uses square-root probabilities within the observed elite interval. Algorithm 2 lower-bound resampling applies to both branches.

In [ ]:
seed_run(BASE_SEED)
population_demo = np.random.uniform(
    sne_problem.lower_bounds, sne_problem.upper_bounds, size=(POP_SIZE, 2),
)
fitness_demo = np.array([sne_problem.compute_fitness(x) for x in population_demo])
elite, elite_copy, random_subset = split_population(
    population_demo, fitness_demo, True,
    sne_problem.lower_bounds, sne_problem.upper_bounds,
)
assert len(elite) == len(elite_copy) == POP_SIZE // 4
assert len(random_subset) == POP_SIZE // 2
print("Subset sizes:", dict(elite=len(elite), copies=len(elite_copy), random=len(random_subset)))
for name, subset in [("random", random_subset), ("elite-copy", elite_copy)]:
    print(name, "qubits per parameter:", int(np.log2(len(subset))))

values = random_subset[:, 1]  # Omega_m illustration
random.seed(BASE_SEED)
unitary = build_dimension_circuit(values, 1., 1., SHOTS, mode="statevector")
random.seed(BASE_SEED)
measured = build_dimension_circuit(values, 1., 1., SHOTS, mode="measure")
names = [instruction.operation.name for instruction in measured.data]
assert names.index("rx") < names.index("measure")
assert "measure" not in [instruction.operation.name for instruction in unitary.data]
print(measured.draw(output="text", fold=-1))

demo_backend = seed_run(BASE_SEED)
counts = demo_backend.run(
    transpile(measured, demo_backend, seed_transpiler=BASE_SEED),
    shots=32768, seed_simulator=BASE_SEED,
).result().get_counts()
observed_probabilities = get_basis_counts(counts, measured.num_qubits) / 32768
expected_probabilities = Statevector(unitary).probabilities()
np.testing.assert_allclose(observed_probabilities, expected_probabilities, atol=0.02)
decoded_random = decode_random_subset(counts, measured.num_qubits, 0., .5, len(values))
assert np.all((decoded_random >= 0.) & (decoded_random <= .5))
elite_qc = build_dimension_circuit(elite_copy[:, 1], 1., 1., SHOTS)
elite_counts = demo_backend.run(transpile(elite_qc, demo_backend), shots=SHOTS).result().get_counts()
box_min, box_max = elite[:, 1].min(), elite[:, 1].max()
decoded_copies = decode_elite_subset(elite_counts, elite_qc.num_qubits,
                                     box_min, box_max, len(elite_copy))
assert np.all((decoded_copies >= box_min) & (decoded_copies <= box_max))
print("Post-gate readout and bounded decoding verified.")

## 6. Run AEQGA on the real calibrated objective

The core records generation zero and then performs exactly MAX_GEN updates. Validate histories, elitism and evaluation counts before interpreting the returned best point. A single demonstration run is not an uncertainty estimate.

In [ ]:
simulator = seed_run(BASE_SEED)
best, populations, bests_log = run_aeqga_dual(sne_problem, params, simulator=simulator)
assert best.evaluations == POP_SIZE * (MAX_GEN + 1)
assert len(populations) == len(bests_log) == MAX_GEN + 1
history_chi2 = np.array([entry[1] for entry in bests_log])
assert np.all(np.diff(history_chi2) <= 1e-8)
assert np.isfinite(best.fitness)
assert abs(best.fitness - sne_problem.compute_fitness(best.x)) < 1e-7
print("Best [H0, Omega_m]:", best.x)
print("Best chi2:", best.fitness)
print("Excess above configured classical reference:",
      best.fitness - configured_reference["chi2"])

## 7. Convergence and likelihood diagnostics

All surface values below come from **the same sne_problem used by the optimizer**. Delta chi-squared is referenced to its independent classical minimum, not the coarsely sampled plotted minimum. Ascending levels 2.30, 6.18 and 11.83 are conventional two-parameter Gaussian likelihood reference levels.

These are likelihood-shape diagnostics, **not the paper's optimizer-scatter contours**. The 100/300-node nearest-neighbor approximation is discontinuous and can visibly quantize narrow features; set DISTANCE_GRID=0 and rerun for direct-distance diagnostics.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
generations = np.arange(len(bests_log))
axes[0].plot(generations, history_chi2 - configured_reference["chi2"], color="#245b8a", lw=2)
axes[0].axhline(0, color="#666666", ls="--", lw=1)
axes[0].set(xlabel="Update (0 = initial population)", ylabel="Best chi2 - classical minimum",
            title=f"Pantheon+ convergence ({MODE})")

h0_values = np.linspace(max(60., configured_reference["H0"]-1.4),
                        min(80., configured_reference["H0"]+1.4), 170)
om_low, om_high = max(0., configured_reference["Omega_m"]-.085), min(.5, configured_reference["Omega_m"]+.085)
omega_values = (sne_problem.omega_grid[(sne_problem.omega_grid >= om_low) &
                                      (sne_problem.omega_grid <= om_high)]
                if DISTANCE_GRID else np.linspace(om_low, om_high, 110))
chi2_surface = sne_problem.objective_grid(h0_values, omega_values)
delta_chi2 = chi2_surface - configured_reference["chi2"]
assert delta_chi2.min() >= -1e-5
levels = [2.30, 6.18, 11.83]
axes[1].contourf(h0_values, omega_values, delta_chi2,
                levels=[0., *levels], colors=["#4389b5", "#8ebbd4", "#d6e6ef"])
contours = axes[1].contour(h0_values, omega_values, delta_chi2,
                          levels=levels, colors="#245b8a", linewidths=1.1)
axes[1].clabel(contours, fmt={2.30: "2.30", 6.18: "6.18", 11.83: "11.83"}, fontsize=8)
axes[1].plot(configured_reference["H0"], configured_reference["Omega_m"],
             "*", color="#152c43", ms=11, label="Classical reference")
# An unconverged demo point may lie outside this zoom: do not expand/tune the likelihood region to it.
if h0_values[0] <= best.x[0] <= h0_values[-1] and omega_values[0] <= best.x[1] <= omega_values[-1]:
    axes[1].plot(*best.x, "o", color="#d26d32", ms=5, label="AEQGA")
else:
    print("AEQGA demo point lies outside the likelihood zoom; inspect convergence before scientific interpretation.")
axes[1].set(xlabel="H0 [km/s/Mpc]", ylabel="Omega_m", title="Calibrated likelihood, not optimizer scatter")
axes[1].legend(frameon=False, fontsize=8)
fig.tight_layout()
figure_path = output_path("png", "notebook_sne_diagnostics.png")
fig.savefig(figure_path)
display(fig)
plt.close(fig)
print("Saved", figure_path)

## 8. Save current results with provenance

Notebook-specific names avoid overwriting the old legacy JSON/figures. Store the selection, redshift convention, grid approximation, parameters, seeds, software versions and data hashes alongside both classical references. Seeds aid auditing, but are not a claim of cross-version bitwise reproducibility.

In [ ]:
run_record = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "experiment": "calibrated_pantheon_plus", "mode": MODE,
    "parameter_order": ["H0", "Omega_m"], "seed": BASE_SEED,
    "versions": VERSIONS, "data_sha256": DATA_PROVENANCE,
    "selection": SELECTION, "redshift": REDSHIFT, "distance_grid": DISTANCE_GRID,
    "parameters": {name: getattr(params, name) for name in [
        "pop_size", "max_gen", "num_shots", "p_cross", "p_mut", "elite_margin",
    ]},
    "classical_reference": configured_reference, "direct_reference": direct_reference,
    "aeqga": {"parameters": best.x.tolist(), "chi2": float(best.fitness),
              "generation": best.gen, "evaluations": best.evaluations},
    "best_history": bests_log,
    "uncertainty_interpretation": "single run; no uncertainty estimate",
}
record_path = output_path("json", "notebook_sne_results.json")
record_path.write_text(json.dumps(run_record, indent=2, allow_nan=False) + "\n")
print("Saved", record_path)

## 9. Optional independent-run ensemble

Set RUN_ENSEMBLE=True explicitly. Each run gets a fresh population and a recorded seed; no restart reuses a previous best fit. The paper configuration is 300 runs × 32 individuals × 50 updates. A small demo ensemble is exploratory.

Report the mean and standard deviation of optimizer outcomes, not posterior uncertainties. Plot actual best-fit points; this avoids the old incorrect Gaussian peak-height/KDE confidence labels and singular KDE failures. No statistical contour is fabricated from a single run.

In [ ]:
ensemble_record = None
if RUN_ENSEMBLE:
    ensemble_points, ensemble_chi2 = [], []
    ensemble_seeds = [BASE_SEED + 1000 + i for i in range(ENSEMBLE_RUNS)]
    for index, seed in enumerate(ensemble_seeds):
        run_best, _, _ = run_aeqga_dual(sne_problem, params, simulator=seed_run(seed))
        ensemble_points.append(run_best.x.tolist())
        ensemble_chi2.append(float(run_best.fitness))
        if (index + 1) % 10 == 0:
            print(f"Completed {index + 1}/{ENSEMBLE_RUNS}")
    ensemble_points = np.asarray(ensemble_points)
    means, stds = ensemble_points.mean(axis=0), ensemble_points.std(axis=0, ddof=0)
    print("Optimizer means [H0, Omega_m]:", means)
    print("Optimizer standard deviations (ddof=0):", stds)
    ensemble_record = {
        "configuration": run_record, "runs": ENSEMBLE_RUNS, "seeds": ensemble_seeds,
        "best_points": ensemble_points.tolist(), "best_chi2": ensemble_chi2,
        "mean": means.tolist(), "std_ddof0": stds.tolist(),
        "interpretation": "optimizer outcome scatter, not posterior uncertainty",
    }
    output_path("json", "notebook_sne_ensemble.json").write_text(
        json.dumps(ensemble_record, indent=2, allow_nan=False) + "\n")
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.scatter(ensemble_points[:, 0], ensemble_points[:, 1], s=18, alpha=.6, color="#245b8a")
    ax.plot(configured_reference["H0"], configured_reference["Omega_m"], "*", ms=12,
            color="#d26d32", label="Classical reference")
    ax.set(xlabel="H0 [km/s/Mpc]", ylabel="Omega_m",
           title=f"Optimizer outcome scatter ({ENSEMBLE_RUNS} independent runs)")
    ax.legend(frameon=False)
    fig.tight_layout()
    fig.savefig(output_path("png", "notebook_sne_optimizer_scatter.png"))
    display(fig)
    plt.close(fig)
else:
    print("Ensemble disabled. No optimizer-scatter uncertainty or contour has been estimated.")

## 10. Optional exact-statevector diagnostic

This calls the shared implementation with exact basis probabilities. Classical fresh draws, random qubit/gate choices and decoding resampling still exist. Do not describe this as a deterministic reproduction of the shot-based experiment.

In [ ]:
if RUN_STATEVECTOR:
    sv_best, sv_populations, sv_history = run_aeqga_sv(
        sne_problem, params, simulator=seed_run(BASE_SEED, statevector=True),
    )
    assert len(sv_history) == MAX_GEN + 1
    assert sv_best.evaluations == POP_SIZE * (MAX_GEN + 1)
    print("Statevector best:", sv_best.x, "chi2:", sv_best.fitness)
else:
    print("Statevector diagnostic disabled.")

## 11. Separate BAO and optional CMB+BAO workflow

BAO is implemented with 16 measurements and the published covariance blocks (11 correlated rows versus the paper's stated 10; see the specification). A BAO-only check below is always inexpensive.

For CMB+BAO, enable RUN_CMB_BAO after fetching the TT/model files. The diagonal Planck TT approximation is **not the official Plik likelihood**. CMB_BACKEND="camb" is a deliberately selected reference backend, not the authors' unidentified PICO model; strict "pico" raises outside its training domain and "hybrid" records CAMB fallbacks. No forced extrapolation or silent fallback is used.

This experiment uses **chi2_CMB + chi2_BAO**, never chi2_SNe + chi2_CMB + chi2_BAO. Its local classical baseline can be slow; all choices and data hashes are saved.

In [ ]:
bao = BAOLikelihood()
bao_check_point = [66.76, 0.323]
print("BAO measurements:", len(bao.z))
print("BAO chi2 at comparison point:", bao.chi2(*bao_check_point))

cmb_record = None
if RUN_CMB_BAO:
    from scipy.optimize import minimize
    cmb_directory = PROJECT_ROOT / "data" / "cmb"
    cmb_inputs = [cmb_directory / "COM_PowerSpect_CMB-TT-full_R3.01.txt"]
    if CMB_BACKEND != "camb":
        cmb_inputs.append(cmb_directory / "pico4_tailmonty_v35_py3.dat")
    if not all(p.is_file() for p in cmb_inputs):
        raise FileNotFoundError("Run python -m scripts.fetch_cmb_data before enabling CMB+BAO.")
    cmb_problem = BAOCMBProblem(
        data_dir=cmb_directory, backend=CMB_BACKEND,
        ell_min=CMB_ELL_MIN, error_mode=CMB_ERROR_MODE,
    )
    cmb_reference = minimize(
        cmb_problem.compute_fitness, [67., .32], method="Nelder-Mead",
        bounds=[(60., 80.), (0., .5)],
        options={"xatol": 1e-5, "fatol": 1e-5, "maxiter": 250},
    )
    if not cmb_reference.success:
        raise RuntimeError("CMB+BAO local baseline failed: " + str(cmb_reference.message))
    cmb_best, cmb_populations, cmb_history = run_aeqga_dual(
        cmb_problem, params, simulator=seed_run(BASE_SEED + 2000),
    )
    assert len(cmb_history) == MAX_GEN + 1
    assert cmb_best.evaluations == POP_SIZE * (MAX_GEN + 1)
    assert np.isfinite(cmb_best.fitness)
    cmb_record = {
        "experiment": "cmb_plus_bao_only", "mode": MODE,
        "parameter_order": ["H0", "Omega_m"], "versions": {**VERSIONS, "camb": version("camb")},
        "parameters": run_record["parameters"], "seed": BASE_SEED + 2000,
        "backend": CMB_BACKEND, "ell_min": CMB_ELL_MIN,
        "ell_max": int(cmb_problem.cmb.ell[-1]), "error_mode": CMB_ERROR_MODE,
        "data_sha256": {p.name: sha256_file(p) for p in cmb_inputs},
        "local_classical_reference": {"parameters": cmb_reference.x.tolist(),
                                       "chi2": float(cmb_reference.fun)},
        "aeqga": {"parameters": cmb_best.x.tolist(), "chi2": float(cmb_best.fitness),
                  "evaluations": cmb_best.evaluations},
        "best_history": cmb_history,
        "camb_fallback_evaluations": cmb_problem.cmb.fallback_evaluations,
        "exact_paper_reproduction": False,
    }
    output_path("json", "notebook_cmb_bao_results.json").write_text(
        json.dumps(cmb_record, indent=2, allow_nan=False) + "\n")
    print(json.dumps(cmb_record["local_classical_reference"], indent=2))
    print("CMB+BAO AEQGA:", cmb_record["aeqga"])
    print("Recorded CAMB fallbacks:", cmb_problem.cmb.fallback_evaluations)
else:
    print("CMB+BAO disabled; no CMB likelihood or combined fit was run.")

## 12. HQGA bridge and optional native comparison

The adapters are retained in `aeqga.integrations.hqga`, not in the AEQGA
evolution loop. They share the calibrated objective in either direction;
they do not mix HQGA's reinforcement gates with AEQGA's amplitude encoding.

The always-run check below verifies Gray-code decoding, objective equality
and round-trip interface conversion on the real Pantheon+ objective. It is
**not a native HQGA optimizer run**. See
[the integration guide](../docs/HQGA_INTEGRATION.md) for upstream connection
examples, parameter mapping, return types and comparison methodology.

To run HQGA, explicitly set RUN_HQGA_COMPARISON=True and supply native
HQGA_PARAMS and HQGA_DEVICE_FEATURES in a compatible environment. The original
HQGA runtime uses old Qiskit APIs; this notebook never installs/downgrades
dependencies or substitutes another algorithm. Choose the HQGA population and
bit depth independently: its total qubits are population × dimensions × bits.
Report grid resolution and actual fitness evaluations, not just generations.


In [ ]:
from aeqga.integrations.hqga import AEQGAProblemForHQGA, HQGAProblemAdapter

HQGA_BITS = 4  # Deliberately coarse contract example, not a precision-fit recommendation
hqga_objective = AEQGAProblemForHQGA(
    sne_problem, num_bit_code=HQGA_BITS, name="Calibrated Pantheon+ [H0, Omega_m]"
)
gray_example = "0" * (hqga_objective.dim * HQGA_BITS)
decoded = hqga_objective.convert(gray_example)
np.testing.assert_allclose(decoded, sne_problem.lower_bounds)
np.testing.assert_allclose(
    hqga_objective.evaluate(gray_example), sne_problem.compute_fitness(decoded)
)
roundtrip = HQGAProblemAdapter(hqga_objective)
np.testing.assert_allclose(roundtrip.compute_fitness(decoded), sne_problem.compute_fitness(decoded))
print("HQGA objective contract verified; grid resolution:", hqga_objective.resolution)
print("Native HQGA optimizer has NOT run unless the explicit opt-in below succeeds.")

hqga_record = None
if RUN_HQGA_COMPARISON:
    if HQGA_PARAMS is None or HQGA_DEVICE_FEATURES is None:
        raise RuntimeError("Supply native HQGA_PARAMS and HQGA_DEVICE_FEATURES; see docs/HQGA_INTEGRATION.md")
    try:
        from HQGA import hqga_utils, hqga_algorithm
    except ImportError as exc:
        raise RuntimeError("Native HQGA requires a compatible execution environment; no fallback is used") from exc
    total_qubits = HQGA_PARAMS.pop_size * hqga_objective.dim * HQGA_BITS
    if total_qubits > 24:
        raise ValueError("This notebook's HQGA smoke guard allows at most 24 qubits; configure a feasible budget")
    random.seed(BASE_SEED)
    np.random.seed(BASE_SEED)
    # Backend seeding remains the supplied device's responsibility.
    native_problem = AEQGAProblemForHQGA(sne_problem, num_bit_code=HQGA_BITS)
    native_circuit = hqga_utils.setupCircuit(
        HQGA_PARAMS.pop_size, native_problem.dim * native_problem.num_bit_code
    )
    from time import perf_counter
    started = perf_counter()
    hbest, hpopulations, hhistory = hqga_algorithm.runQGA(
        HQGA_DEVICE_FEATURES, native_circuit, HQGA_PARAMS, native_problem
    )
    hqga_record = {
        "algorithm": "native HQGA", "objective_provenance": run_record,
        "parameters": {k: v for k, v in vars(HQGA_PARAMS).items()},
        "seed": BASE_SEED, "backend_seed_note": "configured on supplied device",
        "bits_per_dimension": HQGA_BITS, "resolution": native_problem.resolution.tolist(),
        "total_qubits": total_qubits, "evaluations": native_problem.evaluations,
        "elapsed_seconds": perf_counter() - started,
        "best_x": list(hbest.phenotype), "fitness": float(hbest.fitness),
        "best_history": hhistory, "population_history": hpopulations,
        "exact_paper_reproduction": False,
    }
    output_path("json", "notebook_hqga_comparison.json").write_text(
        json.dumps(hqga_record, indent=2, default=lambda value: value.tolist() if isinstance(value, np.ndarray) else str(value))
    )
    print("Native HQGA:", hbest.phenotype, hbest.fitness, "evaluations:", native_problem.evaluations)


## Interpretation and remaining work

- A finite result and monotonic best-history validate execution, not convergence to the paper or a quantum speedup.
- Compare AEQGA chi-squared against the independent classical reference before interpreting parameters.
- Likelihood contours and repeated-optimizer scatter answer different questions; do not use one as the other.
- Demonstration runs do not establish the paper's quoted uncertainties. Full ensembles are opt-in and their actual configuration is recorded.
- Paper-identical CMB+BAO reproduction remains uncertified until the authors' model and TT conventions are identified.

New notebook artifacts use the **notebook_** prefix under outputs/png and outputs/json. They never load the old legacy result files. See [methodology](../docs/TASKS_1_TO_7_REPORT.md), [configuration details and primary data sources](../docs/REPLICATION_SPEC.md), and the root [README](../README.md).